In [ ]:
#import cProfile, pstats, io
#from pstats import SortKey
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
from tqdm.auto import tqdm

from covert_quantum_analysis import FIM, vectorized_QFIM

%matplotlib widget

In [ ]:

α = .7
η1 = .9
η2 = .9
nbar = .1
r = .1
a = .7



In [ ]:
def σ(η1,η2,α,r):
    l1 = [1+2*η1*α**2*np.sinh(r)**2 + 2*nbar, 0, -2*α*np.sqrt((1-α**2)*η1*η2)*np.sinh(r)**2,0, -2*α*np.sqrt(η1)*np.sinh(r)*np.cosh(r),0]
    l2 = [0,1+2*η1*α**2*np.sinh(r)**2 + 2*nbar, 0, -2*α*np.sqrt((1-α**2)*η1*η2)*np.sinh(r)**2,0, 2*α*np.sqrt(η1)*np.sinh(r)*np.cosh(r)]
    l3 = [-2*α*np.sqrt((1-α**2)*η1*η2)*np.sinh(r)**2,0,1+2*η2*(1-α**2)*np.sinh(r)**2+2*nbar,0,2*np.sqrt(η2*(1-α**2))*np.sinh(r)*np.cosh(r),0]
    l4 = [0,-2*α*np.sqrt((1-α**2)*η1*η2)*np.sinh(r)**2,0,1+2*η2*(1-α**2)*np.sinh(r)**2+2*nbar,0,-2*np.sqrt(η2*(1-α**2))*np.sinh(r)*np.cosh(r)]
    l5 = [-2*α*np.sqrt(η1)*np.sinh(r)*np.cosh(r),0,2*np.sqrt(η2*(1-α**2))*np.sinh(r)*np.cosh(r),0,1+2*np.sinh(r)**2,0]
    l6 = [0,2*α*np.sqrt(η1)*np.sinh(r)*np.cosh(r),0,-2*np.sqrt(η2*(1-α**2))*np.sinh(r)*np.cosh(r),0,1+2*np.sinh(r)**2]
    σ = np.array([l1,l2,l3,l4,l5,l6])
    return σ

In [ ]:
def dσdη1(η1,η2,α,r):
    l1 = [2*α**2*np.sinh(r)**2,0, -α*np.sqrt((1-α**2)*η2/η1)*np.sinh(r)**2,0,-α*np.sqrt(1/η1)*np.sinh(r)*np.cosh(r),0]
    l2 = [0,2*α**2*np.sinh(r)**2,0, -α*np.sqrt((1-α**2)*η2/η1)*np.sinh(r)**2,0,α*np.sqrt(1/η1)*np.sinh(r)*np.cosh(r)]
    l3 = [-α*np.sqrt((1-α**2)*η2/η1)*np.sinh(r)**2,0,0,0,0,0]
    l4 = [0,-α*np.sqrt((1-α**2)*η2/η1)*np.sinh(r)**2,0,0,0,0]
    l5 = [-α*np.sqrt(1/η1)*np.sinh(r)*np.cosh(r),0,0,0,0,0]
    l6 = [0,α*np.sqrt(1/η1)*np.sinh(r)*np.cosh(r),0,0,0,0]
    dσdη1 = np.array([l1,l2,l3,l4,l5,l6])
    return dσdη1

In [ ]:
def dσdη2(η1,η2,α,r):
    l1 = [0,0,-α*np.sqrt((1-α**2)*η1/η2)*np.sinh(r)**2,0,0,0]
    l2 = [0,0,0,-α*np.sqrt((1-α**2)*η1/η2)*np.sinh(r)**2,0,0]
    l3 = [-α*np.sqrt((1-α**2)*η1/η2)*np.sinh(r)**2,0,2*(1-α**2)*np.sinh(r)**2,0,np.sqrt((1-α**2)/η2)*np.sinh(r)*np.cosh(r),0]
    l4 = [0,-α*np.sqrt((1-α**2)*η1/η2)*np.sinh(r)**2,0,2*(1-α**2)*np.sinh(r)**2,0,-np.sqrt((1-α**2)/η2)*np.sinh(r)*np.cosh(r)]
    l5 = [0,0,np.sqrt((1-α**2)/η2)*np.sinh(r)*np.cosh(r),0,0,0]
    l6 = [0,0,0,-np.sqrt((1-α**2)/η2)*np.sinh(r)*np.cosh(r),0,0]
    dσdη2 = np.array([l1,l2,l3,l4,l5,l6])
    return dσdη2

In [ ]:
σt = σ(η1,η2,α,r)

In [ ]:
def ClassicalFIE(eta1, eta2, n, a, nbar):
    # Generated via lambdify + inspect on a symbolic expression
    return (1/4)*n*(a*eta1 - np.sqrt(eta1*eta2*(1 - a**2)))*(-a**2*eta2 - a*np.sqrt(eta1*eta2*(1 - a**2)) + eta2)/((2*nbar + 1)*(a*eta1*(-a**2*eta2 - a*np.sqrt(eta1*eta2*(1 - a**2)) + eta2) + eta2*(a**2 - 1)*(a*eta1 - np.sqrt(eta1*eta2*(1 - a**2))))*(a**2*eta1 + a**2*eta2 - eta2))

In [ ]:
def vec_eval(η1s,η2s,αs,rs):
    return vectorized_QFIM(σ,[dσdη1,dσdη2],η1s,η2s,αs,rs)

In [ ]:
def vec_QFIEs(QFIMs,a):
    # Currently only written for 2 parameter estimation
    aeff = a #if (np.abs(a) > .5) else (-a+ np.sign(a))
    Bm1 = [[aeff,0],[np.sqrt(1-aeff**2),1]]
    shape = QFIMs.shape[0:4]
    QFIEs = np.zeros(shape)
    for i1,i2,i3,i4 in tqdm(np.ndindex(shape),total=np.prod(shape),smoothing=.01):
        Qtm1 = np.transpose(Bm1)@np.linalg.inv(QFIMs[i1,i2,i3,i4,:,:])@Bm1
        FIE = 1/Qtm1[0,0]
        QFIEs[i1,i2,i3,i4] = FIE
    return QFIEs

In [ ]:
def eval_settings(η1,η2,α,r,a):
    σval = σ(η1,η2,α,r)
    dσs = [dσdη1(η1,η2,α,r),dσdη2(η1,η2,α,r)]
    FIMval = FIM(σval,dσs)
    return FIE(FIMval,a)

In [ ]:
def save_state(filename):
    np.savez_compressed(filename, eta1s = η1s, eta2s = η2s,alphavals = αvals,rvals=rvals,QFIMs=QFIMs)

In [ ]:
def load_state(filename):
    loaded = np.load(filename)
    return [loaded['eta1s'],loaded['eta2s'],loaded['alphavals'],loaded['rvals'],loaded['QFIMs']]

In [ ]:
η1s = np.linspace(.01,.99,20)
η2s = η1s
αvals = np.linspace(-.991,.99,40)
#rvals = np.logspace(-7,0,3)
rvals = [np.arcsinh(np.sqrt(2)*np.sinh(1e-7))]
#avals = np.linspace(-.99,.99,10)
#with cProfile.Profile() as pr

QFIMs = np.real_if_close(vec_eval(η1s,η2s,αvals,rvals))

#eps = pstats.Stats(pr).sort_stats("cumtime")  # cumulative time by function
#eps.print_stats(20)

#np.save('QFIEs2',QFIEs)

#save_state('../_data/QFIEs3')
#QFIEs = np.load('QFIEs2.npy')
#[η1s, η2s, αvals, rvals, QFIMs] = load_state('../_data/QFIEs3.npz')

In [ ]:
QFIEssingle = vec_QFIEs(QFIMs[:,:,:,0:1,:,:],0)

In [ ]:
[η1grid,η2grid] = np.meshgrid(η1s,η2s)
maximizedSingle = np.nanmax(QFIEssingle,axis=2)[:,:,0]
αtargetarg = np.argmin(np.abs(αvals))
αsingle = αvals[np.nanargmax(QFIEssingle,axis=2)]
maxvscentered = maximizedSingle - QFIEssingle[:,:,αtargetarg]
check = 4*ClassicalFIE(η1grid*(1-η1grid), η2grid*(1-η2grid), np.sinh(rvals[0])**2, 0, nbar)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,maximizedSingle-check)
plt.show()

In [ ]:
#[η1s, η2s, αvals, rvals, QFIMs] = load_state('../_data/QFIEs3.npz')
target = np.sqrt(1/2)
QFIEs = vec_QFIEs(QFIMs,target)
del QFIMs

In [ ]:
np.shape(QFIEs)

In [ ]:
#target = np.sqrt(1/2)
#targetarg = np.argmin(np.abs(avals-target))
targeteddata = QFIEs[:,:,:,0]#,targetarg]
maximized = np.nanmax(targeteddata,axis=2)
[η1grid,η2grid] = np.meshgrid(η1s,η2s)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,maximized)
plt.show()

In [ ]:
zeroval = np.argmin(np.abs(αvals))
maximizedpos = np.nanmax(QFIEs[:,:,zeroval:,0],axis=2)
maximizedneg = np.nanmax(QFIEs[:,:,0:zeroval,0],axis=2)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,maximizedpos)
ax.plot_surface(η1grid,η2grid,maximizedneg)
plt.show()

In [ ]:
CFI = ClassicalFIE(η1grid, η2grid, np.sinh(rvals[0])**2, target, nbar)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,maximized)
ax.plot_surface(η1grid,η2grid,CFI)
plt.show()

In [ ]:
QFIMixed = 4*ClassicalFIE(η1grid*(1-η1grid), η2grid*(1-η2grid), np.sinh(rvals[0])**2, target, nbar)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,maximized)
ax.plot_surface(η1grid,η2grid,QFIMixed)
plt.show()

In [ ]:
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,maximized-QFIMixed)
#ax.plot_surface(η1grid,η2grid,np.zeros(η1grid.shape))

plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,sp.signal.wiener(maximizedpos-maximizedneg,(10,10)))
plt.show()

In [ ]:
αmax = αvals[np.nanargmax(targeteddata,axis=2)]
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,np.abs(αmax))
plt.show()

In [ ]:

split1 = np.argmin(np.abs(.5-η1s))
split2 = np.argmin(np.abs(.5-η2s))

oneddata1 = αmax[split1,:]
oneddata2 = αmax[:,split2]
fig = plt.figure()
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)
ax1.plot(η1s,np.abs(oneddata2))
ax2.plot(η2s,np.abs(oneddata1))
plt.show()

In [ ]:
xdata = np.vstack((η1grid.ravel(), η2grid.ravel()))
ydata = np.abs(αmax).ravel()
p0 = [0,1,-1]
def _logvals(X,x0,l1,l2):
    return x0+l1*np.log(X[0]) + l2*np.log(X[1])
def _jac_logvals(X,x0,l1,l2):
    return np.transpose(np.array([np.ones(X[0].shape),np.log(X[0]),np.log(X[1])]))
popt, pcov = sp.optimize.curve_fit(_logvals, xdata, ydata, p0,jac=_jac_logvals)
fitteddata = popt[0] + popt[1]*np.log(η1grid) + popt[2]*np.log(η2grid)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,fitteddata)
ax.plot_surface(η1grid,η2grid,np.abs(αmax))

plt.show()

In [ ]:
def fitα(fitfunc,data,aval,η1s,η2s,p0,bounds=(-np.inf,np.inf)):
    QFIEs = vec_QFIEs(data[:,:,:,0:1,:,:],aval)
    αmax = np.abs(αvals[np.nanargmax(QFIEs[:,:,:,0],axis=2)])
    η1grid, η2grid = np.meshgrid(η1s,η2s)
    xdata = np.vstack((η1grid.ravel(), η2grid.ravel()))
    ydata = αmax.ravel()
    popt, _ = sp.optimize.curve_fit(fitfunc, xdata, ydata, p0,bounds=bounds)#,jac=_jac_logvals)
    fitteddata = fitfunc(np.array([η1grid,η2grid]),*popt)
    res = αmax-fitteddata
    totalerr = np.sqrt(np.sum(res**2))
    return [*popt,totalerr]



In [ ]:
avals = np.linspace(-.99,.99,1)
def _logvals(X,*args):
    (x0,l1,l2) = args
    return x0+l1*np.log(X[0]) + l2*np.log(X[1])
p0 = [0,1,-1]
[η1s, η2s, αvals, rvals, QFIMs] = load_state('../_data/QFIEs3.npz')
fits = np.array([fitα(_logvals,QFIMs,aval,η1s,η2s,p0) for aval in tqdm(avals)])
fig, axes = plt.subplots(2,2,layout='constrained')
ylabels = [r'$\mu$',r'$k_1$',r'$k_2$',r'$\sigma$']
for i in range(2):
    for j in range(2):
        axes[i,j].plot(avals,fits[:,2*i+j])
        axes[i,j].set_ylabel(ylabels[2*i+j])
plt.show()

In [ ]:
def _logp1vals(X,*args):
    (x0,a,b,c,d) = args
    return x0 +np.sqrt((a*X[0] + b*np.sqrt(X[0]*X[1]))/(c*X[0] + d*X[1]))
p0 = [0,1,1,1,1]
bounds = (0,3)
fits = np.array([fitα(_logp1vals,QFIMs,aval,η1s,η2s,p0,bounds=bounds) for aval in tqdm(avals)])
fig, axes = plt.subplots(3,2,layout='constrained')
ylabels = ['x0','a','b','c','d',r'$\sigma$']
for i in range(3):
    for j in range(2):
        axes[i,j].plot(avals,fits[:,2*i+j])
        axes[i,j].set_ylabel(ylabels[2*i+j])
plt.show()

In [ ]:
print(popt)

In [ ]:
res = np.abs(αmax)-fitteddata
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,res)
plt.show()

In [ ]:
totalerr = np.sum(res**2)
print(totalerr)

In [ ]:
selector = αmax >0
invselector = np.logical_not(selector)
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_trisurf(η1grid[selector],η2grid[selector],αmax[selector])
ax.plot_trisurf(η1grid[invselector],η2grid[invselector],-αmax[invselector])
plt.show()

In [ ]:
# Code to maybe lowpass the data
def butter_lowpass(cutoff, fs, order=5):
    return sp.signal.butter(order, cutoff, fs=fs, btype='low', analog=False)

def butter_lowpass_filter(data, cutoff, fs, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y1 = sp.signal.lfilter(b, a, data,axis=0)
    y2 = sp.signal.lfilter(b, a, y1,axis=1)
    return y2



In [ ]:
maximizedargpos = αvals[np.nanargmax(QFIEssingle[:,:,zeroval:,0],axis=2)+zeroval]
maximizedargneg = αvals[np.nanargmax(QFIEssingle[:,:,0:zeroval,0],axis=2)]
diffval = maximizedargpos+maximizedargneg
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,diffval)
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.plot_surface(η1grid,η2grid,butter_lowpass_filter(diffval,.2,30))
plt.show()